### Maze-Generation
<img src='/files/images/maze_generation.png'>  

Wir betrachten folgendes Spiel auf einem (m x n)-Gitter.
Jedes Gitterfeld ist von 4 Wänden eingeschlossen.  
Der Spieler startet auf Feld (0, 0). Das Spiel endet, wenn er alle Felder besucht hat.
- Der Spieler kann in eine Wand eine Türe einbauen, um auf
  ein noch **unbesuchtes Nachbarfeld** zu gelangen.
- Der Spieler kann alle bestehenden Türe nutzen, um auf ein bereits besuchtes Feld zurückzukehren. 



Wir benutzen eine Variante unseres Algorithmus zum Finden einer Zusammenhangskomponente, um
alle Felder zu besuchen. Wir speichern
alle **frisch besuchten** Felder in einer Liste `stack`.
Der `stack` enthält alle Felder, deren Nachbarn wir noch nicht angeschaut haben.
Alle **besuchten Felder** speichern wir in einer Menge `visited`.

- Zu Beginn ist `stack = [(0, 0)]` und `visited = {(0, 0)}`.
- Solange noch Felder zu besuchen sind, machen wir folgendes:
  - wir **entfernen** das letzte Feld der Liste Stack und gehen auf dieses Feld.
  - wir besuchen **von diesem Feld aus**, in zufälliger Reihenfolge, alle **unbesuchten** Nachbarfelder und
    fügen diese Felder zu `visited` hinzu und hängen sie ans Ende von `stack` an.

Diese Suchstrategie nennt man auch **Tiefensuche** (depth first search). Wir
- bauen wo möglich Türen ein,
- gehen dann durch eine Tür und setzen die Suche von dort aus fort,
- verlängern so lange als möglich einen Hauptsuchpfad.

In [ ]:
import random


def get_random_neighbors(pos, dims, shuffle=True):
    directions = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    if shuffle:
        random.shuffle(directions)

    x, y = pos
    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < dims[0] and 0 <= ny < dims[1]:
            yield (nx, ny)


def get_moves_df(dims, start=(0, 0), shuffle=True):
    stack = [(start)]
    visited = {start}

    while len(visited) < dims[0]*dims[1]:
        pos = stack.pop()
        for npos in get_random_neighbors(pos, dims, shuffle=shuffle):
            if npos in visited:
                continue
            visited.add(npos)
            stack.append(npos)
            yield pos, npos

In [ ]:
list(get_random_neighbors((0, 0), (3, 3)))

In [ ]:
list(get_moves_df(dims=(4, 2)))

In [ ]:
import widget_helpers as W
import grid_helpers as G
from IPython.display import display


def init_canvas(ncol, nrow, dx=20, dy=20):
    canvas = W.get_canvas(width=ncol*dx, height=nrow*dy)
    grid_spec = (0, 0, dx, dy, ncol, nrow)
    G.draw_grid(canvas, grid_spec)

    canvas.stroke_style = 'blue'
    canvas.line_width = 3
    return canvas, grid_spec


def connect(canvas, src, dst, grid_spec):
    p = G.cr2xy(*src, grid_spec, center=True)
    q = G.cr2xy(*dst, grid_spec, center=True)
    canvas.stroke_line(*p, *q)


def draw_maze(ncol, nrow, dx=20, dy=20, shuffle=True):
    canvas, grid_spec = init_canvas(ncol, nrow, dx, dy)
    for src, dst in get_moves_df(dims=(ncol, nrow), shuffle=shuffle):
        connect(canvas, src, dst, grid_spec)
    display(canvas)

In [ ]:
draw_maze(20, 10, shuffle=False)

### Aufgaben
1. Eine alternative Strategie ist **Breitensuche**:
  Statt des letzten Elementes entfernen wir das 1. Element vom Stack. Das führt dazu, dass alle Suchpfade nach Möglichkeit gleich lang gehalten werden.  
Modifiziere `get_moves_df` entsprechend und nenne die Funktion dann `get_moves_bf`.
Benutze dann diese Funktion in `draw_maze`.
Zeichne einige Mazes.

1. Erweitere die Funktion `draw_maze`.
  Sie soll nun einen Dict `go_back` zurück geben. Der Dict speichert, von wo aus ein neues Feld besucht wurde. Nachstehend ein Maze und wie `go_back`  aussehen könnte. Das Startfeld `(0, 0)` hat keinen Vorgänger.

<img src='/files/images/maze_E.png'>  

```
{(0, 0): None,
 (0, 1): (0, 0),
 (1, 0): (0, 0),
 (1, 1): (1, 0),
 (2, 0): (1, 0),
 (2, 1): (2, 0)}   
```
3. Schreibe eine Funktion `get_path_home(dst, go_back)`, die den
Dict `go_back` nutzt, um einen Weg vom Feld `dst` nach `(0, 0)` zu finden.
Obiger Dict beschreibt folgenden Weg von `(2, 1)` nach `(0, 0)`:  
`[(2, 1), (2, 0), (1, 0), (0, 0)]`

5. Modifiziere die Funktion `draw_maze`, so dass nun auch ein Pfad
  von der rechten unteren Ecke zum Feld `(0, 0)` markiert wird.

<img src='/files/images/maze_E1.png'>